## Download Dataset from Roboflow Universe

RF-DETR expects the dataset to be in COCO format. Divide your dataset into three subdirectories: `train`, `valid`, and `test`. Each subdirectory should contain its own `_annotations.coco.json` file that holds the annotations for that particular split, along with the corresponding image files. Below is an example of the directory structure:

```
dataset/
├── train/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
├── valid/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
└── test/
    ├── _annotations.coco.json
    ├── image1.jpg
    ├── image2.jpg
    └── ... (other image files)
```

[Roboflow](https://roboflow.com/annotate) allows you to create object detection datasets from scratch or convert existing datasets from formats like YOLO, and then export them in COCO JSON format for training. You can also explore [Roboflow Universe](https://universe.roboflow.com/) to find pre-labeled datasets for a range of use cases.

In [2]:
import os
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
project = rf.workspace("fyp-vfrgn").project("7-classes-vehicle")
version = project.version(1)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...


### EDA

In [3]:
import json
import os
from collections import Counter

# Set your dataset directory path
dataset_dir = "7-classes-vehicle-1"

def analyze_coco_text_only(dataset_path, split_name):
    json_path = os.path.join(dataset_path, split_name, "_annotations.coco.json")
    
    if not os.path.exists(json_path):
        print(f"⚠️ Annotation file not found for {split_name}: {json_path}")
        return None, None
        
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    # Map category IDs to names
    cat_map = {c['id']: c['name'] for c in data['categories']}
    
    # Count classes
    class_counts = Counter([cat_map[ann['category_id']] for ann in data['annotations']])
    
    # Get image dimensions
    img_dims = [(img['width'], img['height']) for img in data['images']]
    
    return class_counts, img_dims

splits = ['train', 'valid', 'test']
all_sizes = []

print("=" * 60)
print(f"{'DATASET ANALYSIS':^60}")
print("=" * 60)

for split in splits:
    counts, sizes = analyze_coco_text_only(dataset_dir, split)
    
    if counts:
        all_sizes.extend(sizes)
        total_anns = sum(counts.values())
        
        print(f"\n📂 SPLIT: {split.upper()}")
        print(f"   Total Images:      {len(sizes)}")
        print(f"   Total Annotations: {total_anns}")
        print("-" * 40)
        print(f"   {'CLASS NAME':<25} | {'COUNT':<10}")
        print("-" * 40)
        
        # Print counts for each class, sorted by count descending
        for class_name, count in counts.most_common():
            print(f"   {class_name:<25} | {count:<10}")
        print("-" * 40)

# --- Image Size Analysis (Text Only) ---
if all_sizes:
    widths, heights = zip(*all_sizes)
    avg_w = sum(widths) / len(widths)
    avg_h = sum(heights) / len(heights)
    
    print("\n" + "=" * 60)
    print(f"{'IMAGE RESOLUTION STATS':^60}")
    print("=" * 60)
    print(f"   Resolution Range: {min(widths)}x{min(heights)} to {max(widths)}x{max(heights)}")
    print(f"   Average Resolution: {avg_w:.0f}x{avg_h:.0f}")
    print("=" * 60)

                      DATASET ANALYSIS                      

📂 SPLIT: TRAIN
   Total Images:      5819
   Total Annotations: 33129
----------------------------------------
   CLASS NAME                | COUNT     
----------------------------------------
   Pickup                    | 11649     
   Sedan                     | 11522     
   Truck                     | 5527      
   Suv                       | 2267      
   Van                       | 1621      
   Motorcycle                | 416       
   Bus                       | 127       
----------------------------------------

📂 SPLIT: VALID
   Total Images:      727
   Total Annotations: 4264
----------------------------------------
   CLASS NAME                | COUNT     
----------------------------------------
   Pickup                    | 1469      
   Sedan                     | 1453      
   Truck                     | 759       
   Suv                       | 317       
   Van                       | 199       
   Mot

## Train RF-DETR on custom dataset

### Choose the right `batch_size`

Different GPUs have different amounts of VRAM (video memory), which limits how much data they can handle at once during training. To make training work well on any machine, you can adjust two settings: `batch_size` and `grad_accum_steps`. These control how many samples are processed at a time. The key is to keep their product equal to 16 — that’s our recommended total batch size. For example, on powerful GPUs like the A100, set `batch_size=16` and `grad_accum_steps=1`. On smaller GPUs like the T4, use `batch_size=4` and `grad_accum_steps=4`. We use a method called gradient accumulation, which lets the model simulate training with a larger batch size by gradually collecting updates before adjusting the weights.

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

True
12.6
1


In [5]:
# from rfdetr.datasets.aug_config import AUG_AGGRESSIVE  # Built-in option

# custom_aug_config = {
#     "HorizontalFlip": {"p": 0.5},
#     "ShiftScaleRotate": {
#         "shift_limit": 0.05,
#         "scale_limit": 0.2,       # Increased from 0.15 — helps with vehicle size variance
#         "rotate_limit": 10,
#         "p": 0.5
#     },
#     "RandomBrightnessContrast": {
#         "brightness_limit": 0.2,  # Increased from 0.1 — simulates day/night transitions
#         "contrast_limit": 0.2,   # Increased from 0.1
#         "p": 0.4                  # Increased from 0.3
#     },
#     "OneOf": [                    # New: color variation (supported in 1.5.1)
#         {"HueSaturationValue": {"hue_shift_limit": 10, "sat_shift_limit": 20, "val_shift_limit": 20, "p": 0.5}},
#         {"RGBShift": {"r_shift_limit": 15, "g_shift_limit": 15, "b_shift_limit": 15, "p": 0.5}},
#     ],
# }

In [6]:
from rfdetr import RFDETRSmall
from rfdetr.datasets.aug_config import AUG_AGGRESSIVE
model = RFDETRSmall()
history = []

def callback2(data):
	history.append(data)

model.callbacks["on_fit_epoch_end"].append(callback2)

model.train(
    dataset_dir=dataset.location,
    resolution=512,
    epochs=150,
    batch_size=4,
    grad_accum_steps=4,
    lr=5e-5,
    lr_encoder=7.5e-5,
    weight_decay=1e-4,
    aug_config=AUG_AGGRESSIVE,      
    cls_loss_coef=2.0,                 # <-- stronger classification signal
    lr_scheduler="cosine",             # <-- smooth LR decay
    warmup_epochs=1.0,                 # <-- gentle warmup
    drop_path=0.1,                     # <-- regularization
    save_dataset_grids=True,
    progress_bar=True,
    early_stopping=True,
    early_stopping_patience=20,        # <-- more patience with cosine
    early_stopping_min_delta=0.001,
    checkpoint_interval=5,
    output_dir='output_small'
)

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


[2026-04-11 13:40:16] [INFO] rf-detr - File rf-detr-small.pth already exists with correct MD5 hash.


[2026-04-11 13:40:16] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-11 13:40:16] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-04-11 13:40:17] [INFO] rf-detr - File rf-detr-small.pth already exists with correct MD5 hash.


W0411 13:40:21.893000 11920 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[2026-04-11 13:40:22] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-11 13:40:22] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


...

Val — Overall Metrics                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.5919 │ 0.8780 │ 0.6962 │ 0.7117 │ 0.8112 │ 0.7797 │ 0.8476 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘

                    Val — Per-class Metrics                     
┏━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class      ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ Bus        │   0.7070 │ 0.7444 │ 0.8947 │    0.8500 │ 0.9444 │
│ Motorcycle │   0.4268 │ 0.5633 │ 0.7234 │    0.7556 │ 0.6939 │
│ Pickup     │   0.6210 │ 0.7201 │ 0.8463 │    0.7873 │ 0.9149 │
│ Sedan      │   0.5949 │ 0.7106 │ 0.8393 │    0.7903 │ 0.8947 │
│ Suv        │   0.4780 │ 0.7297 │ 0.6604 │    0.6553 │ 0.6656 │
│ Truck      │   0.6906 │ 0.7909 │ 0.8520 │    0.8007 │ 0.9104 │
│ Van        │   0.6247 │ 0.7226 │ 0.8619 │    0.8190 │ 0.9095 │
└────────────┴──────────┴────────┴────────┴───────────┴────────┘

Epoch 121: 100%|██████████| 1456/1456 [07:25<00:00,  3.27it/s, train/lr=4.23e-6, train/lr_min=1.37e-7, train/lr_max=4.23e-6, val/loss=4.340, val/mAP_50_95=0.592, val/mAP_50=0.878, val/ema_mAP_50_95=0.591, val/F1=0.811, train/loss=4.460]

Monitored metric __rfdetr_effective_map__ did not improve in the last 20 records. Best score: 0.592. Signaling Trainer to stop.


Epoch 121: 100%|██████████| 1456/1456 [07:28<00:00,  3.25it/s, train/lr=4.23e-6, train/lr_min=1.37e-7, train/lr_max=4.23e-6, val/loss=4.340, val/mAP_50_95=0.592, val/mAP_50=0.878, val/ema_mAP_50_95=0.591, val/F1=0.811, train/loss=4.460]
[2026-04-12 04:52:54] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.5927, ema=0.5931)


In [ ]:
# from PIL import Image

# Image.open("output_small/metrics_plot.png")

FileNotFoundError: [Errno 2] No such file or directory: 'output_small/metrics_plot.png'

## Evaluate Fine-tuned RF-DETR Model

Before benchmarking the model, we need to load the best saved checkpoint. To ensure it fits on the GPU, we first need to free up GPU memory. This involves deleting any remaining references to previously used objects, triggering Python’s garbage collector, and clearing the CUDA memory cache.

In [9]:
import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):

    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    # Ensure we drop all strong references
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

In [10]:
cleanup_gpu_memory(model, verbose=True)

[Before] Allocated: 121.69 MB | Reserved: 212.00 MB
[WARNING] Object not fully garbage collected yet.
[After]  Allocated: 0.00 MB | Reserved: 0.00 MB


We load the best-performing model from the `checkpoint_best_total.pth` file using the `RFDETRMedium` class. This checkpoint contains the trained weights from our most successful training run. After loading, we call `optimize_for_inference()`, which prepares the model for efficient inference.

In [11]:
from rfdetr import RFDETRSmall
model = RFDETRSmall(pretrain_weights="output_small/checkpoint_best_total.pth")
model.optimize_for_inference()

[2026-04-12 13:44:46] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-12 13:44:46] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-12 13:44:46] [WARNING] rf-detr - Checkpoint has 8 classes but model is configured for 90. Using checkpoint class count (8). Pass num_classes=8 to suppress this warning.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [12]:
import supervision as sv

ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset.location}/test",
    annotations_path=f"{dataset.location}/test/_annotations.coco.json",
)

In [ ]:
import supervision as sv
import numpy as np
from PIL import Image
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision

targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path)
    detections = model.predict(image, threshold=0)

    # Remove image-level metadata that supervision can't per-detection index
    if 'source_shape' in detections.data:
        del detections.data['source_shape']

    targets.append(annotations)
    predictions.append(detections)

100%|██████████| 728/728 [00:21<00:00, 34.24it/s]


In [18]:
map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()
print(map_result)

Average Precision (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.587
Average Precision (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.882
Average Precision (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.698
Average Precision (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.290
Average Precision (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.583
Average Precision (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.714
